# NB02 — Condroz pilot catchment (approximately 250 ha)
Reproducibly select a DEM-defined agricultural headwater near Ciney, retrieve SPW parcel/soil/infrastructure vectors, generate model-ready raster inputs and run annual + CN smoke tests. **Pilot only:** Copernicus GLO-30 surface elevation and ESA WorldCover 2021 are public numerical fallback sources, not the Walloon LiDAR/WALOUS products. Constant K and generic HSG B are explicitly provisional; no calibrated prediction. Search area is a local Condroz pilot window, not a random sample of the entire Condroz.

## NB02.01 — Setup

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Watem')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/condroz_pilot'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_condroz_runs') / RUN_ID
WORK.mkdir(parents=True)
def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str,cmd)), check=True, **kwargs)
print(ROOT, sys.version, platform.machine())
PACKAGE=ROOT/'data/catchments/condroz_pilot'
PACKAGE.mkdir(parents=True,exist_ok=True)
SEED=2270; RES=20; YEAR=2025
# local search window near Ciney; bounds chosen to stay in one DEM tile
SEARCH_LONLAT=(5.00,50.25,5.16,50.36)
TARGET_HA=250; TOL_HA=25
# Optional locally cached numerical SPW raster paths; do not use rendered WMS/MapServer imagery.
SPW_DEM=None
SPW_WALOUS=None # requires explicit class crosswalk; not implemented here
run([sys.executable,'-m','pip','install','pysheds'])

## NB02.02 — Software and reference configuration

In [ ]:
import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','git-lfs','saga'])
run(['git','lfs','install'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
local_deb = Path('/tmp') / saga_name
shutil.copy2(saga_deb, local_deb)
run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
print('SAGA executable:', saga_cmd)
print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    print(msg[:3500])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower(), f'Missing/incompatible SAGA library: {args}'
run(['ldd',str(binary)])
if not REPO.exists():
    run(['git','clone','https://github.com/watem-sedem/pywatemsedem.git',REPO],
        env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run(['git','-C',REPO,'checkout',REV],env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
print('pywatemsedem',pywatemsedem.__version__)

In [ ]:
required = ['dtm.tif','K3.tif','basemap_landuse.tif','userchoices.ini']
stems = ['catchm_langegracht','river','infrastructure','roads','water','parcels','grass_strips','buffers','force_routing','outlets']
source = REPO/'tests/data'
names = sorted(p.name for p in source.iterdir() if p.is_file() and (p.name in required or p.stem in stems))
assert set(required).issubset(names)
for stem in stems:
    assert all(stem+ext in names for ext in ('.shp','.shx','.dbf','.prj')), stem
def expected_hash(name):
    raw = subprocess.check_output(['git','-C',str(REPO),'show',f'{REV}:tests/data/{name}'])
    if raw.startswith(b'version https://git-lfs'):
        return next(x.split(':',1)[1] for x in raw.decode().splitlines() if x.startswith('oid sha256:'))
    return hashlib.sha256(raw).hexdigest()
hashes = {name:expected_hash(name) for name in names}
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

def valid_file(path, name):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == hashes[name]

def valid(folder):
    return all(valid_file(folder/n,n) for n in names)

DATA.mkdir(parents=True,exist_ok=True)
missing = [n for n in names if not valid_file(DATA/n,n)]
print(f'Valid cached files: {len(names)-len(missing)}/{len(names)}')
if missing:
    # Git LFS pull can fail at a detached HEAD or due to upstream LFS service.
    # Capture stderr in the notebook, rather than only showing CalledProcessError.
    pending = [n for n in missing if not valid_file(source/n,n)]
    if pending:
        p = subprocess.run(['git','-C',str(REPO),'lfs','pull','origin',
                            '--include='+','.join('tests/data/'+n for n in pending)],
                           capture_output=True,text=True)
        print('Git LFS return code:',p.returncode)
        print(p.stdout[-4000:])
        print(p.stderr[-6000:])
    def retrieve(name):
        if valid_file(DATA/name,name):
            return name,'cached'
        if valid_file(source/name,name):
            shutil.copy2(source/name,DATA/name)
            return name,'git-lfs'
        # GitHub media endpoint serves actual LFS objects rather than pointers.
        # Always require the exact SHA256 from the pinned upstream Git tree.
        errors=[]
        tmp=DATA/(name+'.part')
        for base in (
            f'https://media.githubusercontent.com/media/watem-sedem/pywatemsedem/{REV}/tests/data/',
            f'https://github.com/watem-sedem/pywatemsedem/raw/{REV}/tests/data/',
            f'https://raw.githubusercontent.com/watem-sedem/pywatemsedem/{REV}/tests/data/',
        ):
            for attempt in range(2):
                try:
                    req=urllib.request.Request(base+name,headers={'User-Agent':'Watem-Colab/1.0'})
                    with urllib.request.urlopen(req,timeout=60) as response, tmp.open('wb') as dst:
                        shutil.copyfileobj(response,dst)
                    if not valid_file(tmp,name):
                        raise ValueError('SHA256 mismatch (possibly a Git LFS pointer)')
                    tmp.replace(DATA/name)
                    return name,'direct download'
                except Exception as e:
                    errors.append(f'{base}: {type(e).__name__}: {e}')
                    tmp.unlink(missing_ok=True)
                    if attempt==0: time.sleep(2)
        raise RuntimeError(name+': '+' | '.join(errors))
    failures=[]
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures={pool.submit(retrieve,name):name for name in missing}
        for f in as_completed(futures):
            try:
                print(*f.result())
            except Exception as e:
                failures.append(str(e))
                print('FAILED:',e)
    if failures:
        raise RuntimeError(f'{len(failures)} data downloads failed. Valid files are retained on Drive. See HTTP/LFS diagnostics above.')
assert valid(DATA), 'Tutorial data checksum validation failed'
(DATA/'source_manifest.json').write_text(json.dumps({'revision':REV,'sha256':hashes},indent=2))
inputdata_folder = WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
for n in names: shutil.copy2(DATA/n,inputdata_folder/n)
print(f'Validated {len(names)} files in {DATA}')
print('Local working data:', inputdata_folder)

## NB02.03 — Numerical raster download and aligned grid
Use public cloud-optimized GeoTIFFs via range reads. A 20 m grid resamples the 30 m Copernicus DEM, not a gain in native detail. Replace DEM with a numerical Walloon LiDAR raster for real applications.

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, rasterio, requests, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin, xy
from rasterio.features import shapes, rasterize
from shapely.geometry import shape, Point, box
from pysheds.grid import Grid
from IPython.display import display
CRS='EPSG:31370'
bb=transform_bounds('EPSG:4326',CRS,*SEARCH_LONLAT,densify_pts=21)
xmin,ymin=np.floor(np.array(bb[:2])/RES)*RES; xmax,ymax=np.ceil(np.array(bb[2:])/RES)*RES
# buffer around candidate domain; still within tile except possibly west edge; mosaic both adjacent tiles
xmin-=2000; ymin-=2000; xmax+=2000; ymax+=2000
width,height=int((xmax-xmin)/RES),int((ymax-ymin)/RES)
tr=from_origin(xmin,ymax,RES,RES)
profile=dict(driver='GTiff',height=height,width=width,count=1,crs=CRS,transform=tr,nodata=-9999,compress='deflate')
def warp_sources(urls,method):
    dest=np.full((height,width),-9999,dtype='float32')
    for url in urls:
        tmp=np.full_like(dest,-9999)
        with rasterio.open(url) as ds:
            reproject(rasterio.band(ds,1),tmp,dst_transform=tr,dst_crs=CRS,dst_nodata=-9999,resampling=method)
        dest[tmp!=-9999]=tmp[tmp!=-9999]
    assert np.isfinite(dest).all() and (dest!=-9999).all(),'Incomplete raster coverage'
    return dest
def dem_url(lon):
    tile=f'Copernicus_DSM_COG_10_N50_00_E{lon:03d}_00_DEM'
    return f'https://copernicus-dem-30m.s3.amazonaws.com/{tile}/{tile}.tif'
dem_sources=[str(SPW_DEM)] if SPW_DEM else [dem_url(4),dem_url(5)]
wc_url='https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/ESA_WorldCover_10m_2021_v200_N48E003_Map.tif'
dem=warp_sources(dem_sources,Resampling.bilinear)
lc=warp_sources([wc_url],Resampling.nearest).astype('int16')
def write_raster(path,a,dtype='float32',transform=tr):
    p=profile.copy(); p.update(dtype=dtype,height=a.shape[0],width=a.shape[1],transform=transform)
    with rasterio.open(path,'w',**p) as ds: ds.write(a.astype(dtype),1)
write_raster(WORK/'search_dem.tif',dem)
print('Grid:',dem.shape,'Elevation range:',dem.min(),dem.max(),'Land cover:',np.unique(lc))

## NB02.04 — Select a real drainage basin
Hydrologically condition the DEM, calculate D8 accumulation, and randomly choose among candidate outlets with 225–275 ha contributing area. Require an interior catchment and at least 50% WorldCover cropland/grassland. Save the chosen outlet and selection seed.

In [ ]:
# Compatibility with NumPy >= 2.4: pysheds still uses the removed np.in1d.
# Preserve the old one-dimensional return shape and signature.
if not hasattr(np, 'in1d'):
    def _in1d(ar1, ar2, assume_unique=False, invert=False):
        return np.isin(np.asarray(ar1).ravel(), np.asarray(ar2).ravel(),
                       assume_unique=assume_unique, invert=invert)
    np.in1d = _in1d
    print('Applied pysheds / NumPy in1d compatibility shim.')
grid=Grid.from_raster(str(WORK/'search_dem.tif'))
z=grid.read_raster(str(WORK/'search_dem.tif'))
filled=grid.fill_depressions(grid.fill_pits(z))
inflated=grid.resolve_flats(filled)
fdir=grid.flowdir(inflated)
acc=grid.accumulation(fdir)
acc_a=np.asarray(acc)
# candidate outlet within original search bbox (not the surrounding buffer)
rr,cc=np.indices(dem.shape)
xx=xmin+(cc+.5)*RES; yy=ymax-(rr+.5)*RES
candidate=np.argwhere((acc_a*RES**2/10000>=TARGET_HA-TOL_HA)&(acc_a*RES**2/10000<=TARGET_HA+TOL_HA)&(xx>=bb[0])&(xx<=bb[2])&(yy>=bb[1])&(yy<=bb[3]))
rng=np.random.default_rng(SEED); rng.shuffle(candidate)
chosen=None
for row,col in candidate[:500]:
    m=np.asarray(grid.catchment(x=int(col),y=int(row),fdir=fdir,xytype='index'),dtype=bool)
    if not m.any() or m[0,:].any() or m[-1,:].any() or m[:,0].any() or m[:,-1].any(): continue
    area=m.sum()*RES**2/10000
    if abs(area-TARGET_HA)>TOL_HA or np.mean(np.isin(lc[m],[30,40]))<.5: continue
    chosen=(int(row),int(col),m); break
assert chosen is not None,'No acceptable basin: enlarge search window or area tolerance'
row,col,mask=chosen
outxy=xy(tr,row,col)
boundary=gpd.GeoDataFrame(geometry=[shape(g) for g,v in shapes(mask.astype('uint8'),mask=mask,transform=tr) if v==1],crs=CRS).dissolve()
outlet=gpd.GeoDataFrame(geometry=[Point(*outxy)],crs=CRS)
print('Chosen area ha:',mask.sum()*RES**2/10000,'outlet lon/lat:',outlet.to_crs(4326).geometry.iloc[0])
boundary.to_file(PACKAGE/'catchment.shp'); outlet.to_file(PACKAGE/'outlet.gpkg',driver='GPKG')
fig,ax=plt.subplots(figsize=(8,7)); boundary.boundary.plot(ax=ax,color='black'); outlet.plot(ax=ax,color='red'); ax.set_title('Selected Condroz drainage basin'); plt.show()

## NB02.05 — Public SPW vector APIs
Query object IDs first to avoid truncation, then download in batches. Keep source fields and report schema. Failed service downloads stop processing rather than silently substitute fabricated features.

In [ ]:
BASE='https://geoservices.wallonie.be/arcgis/rest/services'
sources={'parcels':f'{BASE}/AGRICULTURE/SIGEC_PARC_AGRI_{YEAR}/MapServer/4','soils':f'{BASE}/SOL_SOUS_SOL/CNSW/MapServer/2','roads':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/21','buildings':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/11','river':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/35','water':f'{BASE}/TOPOGRAPHIE/PICC_VDIFF/MapServer/37'}
session=requests.Session()
def request_json(url,params):
    response=session.get(url,params=params,timeout=120); response.raise_for_status()
    j=response.json()
    if 'error' in j: raise RuntimeError(j['error'])
    return j
def fetch_features(url):
    meta=request_json(url,{'f':'json'})
    assert meta.get('geometryType'),f'Not a feature layer: {url}'
    bounds=boundary.total_bounds
    spatial=dict(where='1=1',geometry=','.join(map(str,bounds)),geometryType='esriGeometryEnvelope',inSR=31370,spatialRel='esriSpatialRelIntersects',returnIdsOnly='true',f='json')
    ids=request_json(url+'/query',spatial).get('objectIds') or []
    features=[]
    for i in range(0,len(ids),100):
        batch=request_json(url+'/query',dict(objectIds=','.join(map(str,ids[i:i+100])),outFields='*',outSR=4326,returnGeometry='true',f='geojson'))
        assert not batch.get('exceededTransferLimit'),'Truncated batch'
        features.extend(batch['features'])
    assert len(features)==len(ids),(len(features),len(ids))
    if not features: return gpd.GeoDataFrame(geometry=[],crs=CRS),meta
    g=gpd.GeoDataFrame.from_features(features,crs=4326).to_crs(CRS)
    g.geometry=g.geometry.make_valid()
    return gpd.clip(g,boundary,keep_geom_type=True),meta
vectors={}; metadata={}
for name,url in sources.items():
    print('Fetching',name,flush=True)
    vectors[name],metadata[name]=fetch_features(url)
    print(name,len(vectors[name]),list(vectors[name].columns))
    if len(vectors[name]): vectors[name].to_file(PACKAGE/f'{name}_source.gpkg',driver='GPKG')
    (PACKAGE/f'{name}_schema.json').write_text(json.dumps(metadata[name],indent=2))
assert len(vectors['parcels']) and len(vectors['soils'])
display(vectors['parcels'].drop(columns='geometry').head())
display(vectors['soils'].drop(columns='geometry').head())

## NB02.06 — Explicit pilot parameter tables
Actual SIGEC attributes are retained. The crop field is discovered from schema; if numeric codes only, join the official code list before assigning crop-specific values. Unknown crops remain visibly flagged and receive an illustrative 0.37 in the pilot. K=30 kg h/(MJ mm) and HSG B are provisional constants, not derived from soil polygons. Annual C factors below are illustrative long-term values based on the Walloon GISER/NVE summary; using them for an event is an explicit teaching approximation.

In [ ]:
p=vectors['parcels'].copy()
fields=metadata['parcels'].get('fields',[])
crop_fields=[f['name'] for f in fields if any(k in (f.get('name','')+' '+f.get('alias','')).lower() for k in ['culture','crop'])]
print('Candidate crop fields:',crop_fields)
# User can set CROP_FIELD explicitly after viewing source schema.
CROP_FIELD=next((f for f in crop_fields if any(x in f.lower() for x in ['lib','desc','label','nom'])),crop_fields[0] if crop_fields else None)
assert CROP_FIELD is not None,'Identify official crop attribute in schema'
# Decode coded value domain where provided
field=next(f for f in fields if f['name']==CROP_FIELD)
domain={str(x['code']):x['name'] for x in (field.get('domain') or {}).get('codedValues',[])}
p['crop_label']=p[CROP_FIELD].astype(str).map(lambda x:domain.get(x,x))
def crop_params(label):
    x=label.lower()
    for keys,c,cn,cl in [(['prairie','herbe','grass'],.01,61,'grass'),(['maïs','mais','maize'],.61,81,'maize'),(['pomme de terre','potato'],.64,81,'potato'),(['betterave','beet'],.41,81,'beet'),(['colza','rapeseed'],.38,81,'rape'),(['froment','orge','épeautre','avoine','seigle','triticale','céréale','cereal'],.39,76,'cereal'),(['lin','chanvre'],.68,81,'flax_hemp')]:
        if any(k in x for k in keys): return c,cn,cl
    return .37,81,'UNKNOWN'
p[['C_crop','CN_base','crop_group']]=pd.DataFrame([crop_params(x) for x in p.crop_label],index=p.index)
p['LANDUSE']=np.where(p.crop_group=='grass',-4,-9999).astype(int)
p['C_reduct']=0.0
p['parcel_id']=np.arange(1,len(p)+1)
p['area_ha']=p.geometry.area/10000
display(p.groupby(['crop_label','crop_group','C_crop','CN_base'],dropna=False).area_ha.agg(['count','sum']).reset_index())
p.drop(columns='geometry').to_csv(PACKAGE/'parcel_parameters.csv',index=False)
p.to_file(PACKAGE/'parcels.gpkg',driver='GPKG')
p[['LANDUSE','C_crop','C_reduct','parcel_id','geometry']].to_file(PACKAGE/'parcels.shp')
if (p.crop_group=='UNKNOWN').any(): print('WARNING: unknown crop labels use provisional C/CN; inspect CSV before scientific use.')
# Crop and management tables for future student interface
pd.DataFrame([('baseline',1,0),('CN_only',1,-10),('C_only',.6,0),('combined',.6,-10)],columns=['scenario','C_multiplier','CN_delta']).to_csv(PACKAGE/'management_scenarios.csv',index=False)

## NB02.07 — Export aligned model inputs and visual QA
Crop polygons override WorldCover within declared fields. Background land-cover coding follows WaTEM/SEDEM. No mapped river is not equivalent to a physical stream: stop before model execution if absent. Preserve soil polygons and provisional K/HSG assumptions.

In [ ]:
# crop domain subset with 2-cell margin, retaining grid alignment
r0,r1=np.where(mask)[0].min()-2,np.where(mask)[0].max()+3
c0,c1=np.where(mask)[1].min()-2,np.where(mask)[1].max()+3
sl=np.s_[r0:r1,c0:c1]; subtr=tr*rasterio.Affine.translation(c0,r0); m=mask[sl]; cover=lc[sl]
# WaTEM landuse codes: 10 generic crop, -3 forest, -4 pasture, -5 pond, -2 infrastructure
bg=np.full(m.shape,10,dtype='int16')
for k,v in {10:-3,20:-4,30:-4,40:10,50:-2,60:10,70:10,80:-5,90:-4,95:-4,100:-4}.items(): bg[cover==k]=v
bg[~m]=-9999
write_raster(PACKAGE/'dtm.tif',dem[sl],transform=subtr)
write_raster(PACKAGE/'basemap_landuse.tif',bg,'int16',subtr)
write_raster(PACKAGE/'K3.tif',np.where(m,30,-9999),'int16',subtr)
write_raster(PACKAGE/'HSG_provisional.tif',np.where(m,2,-9999),'int16',subtr)
for name in ['roads','buildings','river','water']:
    g=vectors[name]
    if len(g):
        g=g[['geometry']].copy()
        if name=='roads': g['paved']=-2
        g.to_file(PACKAGE/(('infrastructure' if name=='buildings' else name)+'.shp'))
fig,ax=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
for a,data,title in [(ax[0,0],dem[sl],'Elevation (m)'),(ax[0,1],bg,'Background land cover'),(ax[0,2],np.where(m,30,np.nan),'Provisional K')]:
    im=a.imshow(np.where(m,data,np.nan)); a.set_title(title); fig.colorbar(im,ax=a,shrink=.7)
p.plot(ax=ax[1,0],column='C_crop',legend=True,vmin=0,vmax=1); ax[1,0].set_title('Parcel C factor')
vectors['soils'].plot(ax=ax[1,1],facecolor='none',edgecolor='black'); ax[1,1].set_title('Actual CNSW soil polygons')
boundary.boundary.plot(ax=ax[1,2],color='black')
for name,color in [('roads','grey'),('river','blue'),('buildings','red')]:
    if len(vectors[name]): vectors[name].plot(ax=ax[1,2],color=color)
ax[1,2].set_title('PICC infrastructure and drainage')
fig.savefig(PACKAGE/'input_atlas.png',dpi=180); plt.show()
manifest=dict(seed=SEED,year=YEAR,area_ha=float(m.sum()*RES**2/10000),outlet_lonlat=list(outlet.to_crs(4326).geometry.iloc[0].coords)[0],resolution=RES,dem=dem_sources,landcover=wc_url,vector_services=sources,K_provisional=30,HSG_provisional='B',unknown_crop_count=int((p.crop_group=='UNKNOWN').sum()),scientific_status='pilot assumptions; not calibrated')
(PACKAGE/'metadata.json').write_text(json.dumps(manifest,indent=2))
print('Saved prepared inputs:',PACKAGE)

## NB02.08 — Build native WaTEM/SEDEM model inputs
Reuse the tested wrapper and reference model choices. Event C is provisional. This prepares native INIs and rasters for annual and CN scenarios; optional smoke runs are separated from preparation.

In [ ]:
from pywatemsedem.catchment import Catchment
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
import configparser
assert (PACKAGE/'river.shp').exists(),'No mapped river in selected basin; choose another seed or implement verified flow-path network.'
catchment=Catchment('condroz_pilot',PACKAGE/'catchment.shp',PACKAGE/'dtm.tif',RES,31370,-9999,results_folder=WORK)
catchment.kfactor=PACKAGE/'K3.tif'
catchment.landuse=PACKAGE/'basemap_landuse.tif'
catchment.vct_river=PACKAGE/'river.shp'
for prop,name in [('vct_infrastructure_buildings','infrastructure'),('vct_infrastructure_roads','roads'),('vct_water','water')]:
    path=PACKAGE/f'{name}.shp'
    if path.exists(): setattr(catchment,prop,path)
default_choices=inputdata_folder/'userchoices.ini'
def settings():
    o=Options(); o.read_values_from_ini(default_choices)
    par=Parameters(); par.read_values_from_ini(default_choices)
    ext=Extensions(); ext.read_values_from_ini(default_choices)
    out=Output(); out.read_values_from_ini(default_choices)
    ep=ExtensionsParameters(ext); ep.apply_defaults()
    for v in (ep.ktc_low,ep.ktc_high,ep.ktc_limit): v.read_value_from_ini(default_choices)
    return Choices(o,par,ext,ep,out)
rain=np.array([0,2,4,7,9,8,5,3,2,0],dtype=float)
rainfile=WORK/'storm.txt'; np.savetxt(rainfile,np.column_stack([np.arange(len(rain))*10,rain]),fmt=['%d','%.3f'],delimiter='\t')
intensity=rain[1:]*6; R=float(np.sum(.29*(1-.72*np.exp(-.05*intensity))*rain[1:])*np.max(np.convolve(rain[1:],np.ones(3),'valid'))*2)
scenarios={}
for nr,mode in enumerate(['annual','event'],1):
    s=Scenario(catchment,YEAR,nr,settings())
    if mode=='event':
        s.choices.extensions.curve_number=True; s.choices.extensions.convert_output=False
        s.choices.parameters.r_factor=R
        ep=s.choices.extensionparameters; ep.alpha=0.; ep.beta=0.; ep.antecedent_rainfall=0.; ep.stream_velocity=.2; ep.desired_timestep=60; ep.endtime_model=720
        s.choices.output.write_total_runoff=True; s.choices.output.write_rainfall_excess=True
    s.vct_parcels=PACKAGE/'parcels.shp'
    s.composite_landuse=s.create_composite_landuse(); s.cfactor=s.create_cfactor()
    ep=s.choices.extensionparameters
    s.ktc=s.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,not s.choices.extensions.create_ktc_map.value)
    if mode=='event':
        lu=np.asarray(s.composite_landuse.arr); inside=np.asarray(catchment.mask.arr)==1
        cn=np.zeros(lu.shape,dtype='float32')
        cn[inside & (lu>0)]=81
        # Use model grid profile, not the source raster shape.
        pr=s.rp.rasterio_profile
        crop_cn=rasterize([(g,float(v)) for g,v in zip(p.geometry,p.CN_base)],out_shape=lu.shape,transform=pr['transform'],fill=0,dtype='float32')
        use=inside & (lu>0)&(crop_cn>0); cn[use]=crop_cn[use]
        for k,v in {-1:100,-2:98,-3:55,-4:61,-5:100,-6:61}.items(): cn[inside & (lu==k)]=v
        assert np.all((cn[inside]>0)&(cn[inside]<=100)),np.unique(lu[inside & (cn==0)])
        s.cn=cn; s.rainfall=rainfile
    s.prepare_input_files(); s.create_ini_file()
    if mode=='event':
        cfg=configparser.ConfigParser(interpolation=None); cfg.optionxform=str; cfg.read(s.ini)
        sec=next(x for x in cfg.sections() if x.lower()=='parameters extensions')
        for key,value in [('Endtime model',720),('Bulk density',int(s.choices.parameters.bulk_density.value))]:
            actual=next((x for x in cfg[sec] if x.lower()==key.lower()),key); cfg[sec][actual]=str(value)
        with s.ini.open('w') as f: cfg.write(f)
    scenarios[mode]=s
    print(mode,s.ini)
DEST=OUTPUT/RUN_ID; DEST.mkdir(exist_ok=True)
for mode,s in scenarios.items(): shutil.copytree(s.sfolder.scenario_folder,DEST/mode,dirs_exist_ok=True)
print('Native inputs copied; INI paths refer to current local runtime. Regenerate on next runtime.')

## NB02.09 — Optional smoke test and basic output reports
Only run after reviewing input atlas and parameter warnings. The results are technical pilot outputs, not a quantitative management assessment.

In [ ]:
RUN_SMOKE=True
if RUN_SMOKE:
    for mode,s in scenarios.items():
        start=time.perf_counter()
        proc=subprocess.run([str(binary),str(s.ini)],capture_output=True,text=True,errors='replace')
        (s.sfolder.scenario_folder/'model.log').write_text(proc.stdout+'\n'+proc.stderr)
        shutil.copytree(s.sfolder.scenario_folder,DEST/mode,dirs_exist_ok=True)
        print(mode,'elapsed s',time.perf_counter()-start,proc.stdout[-3000:],proc.stderr[-1000:])
        proc.check_returncode()
        report=s.sfolder.wsoutput_folder/'Total sediment.txt'
        print(report.read_text(errors='replace'))
        if mode=='event':
            print((s.sfolder.wsoutput_folder/'Total discharge.txt').read_text(errors='replace'))
        with rasterio.open(s.sfolder.wsoutput_folder/'WATEREROS (kg per gridcel).rst') as ds:
            a=ds.read(1,masked=True)*10/RES**2
        fig,ax=plt.subplots(figsize=(7,6)); lim=max(float(np.percentile(np.abs(a.compressed()),98)),.001)
        im=ax.imshow(a,cmap='RdBu',vmin=-lim,vmax=lim); fig.colorbar(im,ax=ax,label='t/ha/'+('year' if mode=='annual' else 'event')); ax.set_title(mode+' | net redistribution'); plt.show()
print('Pilot package:',PACKAGE,'run:',DEST)